In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import mainStartingFive, teamStarPlayer, projectedStartingFive

### Load Model

In [2]:
model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MODEL.pkl')
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/myenv/lib/python3.13/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.7.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, edge_threshold=0.20, stake=10, 
                     variance_inflation=1.1, distribution_type='t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets_{today}.csv', index=False)
singleBets.head()

Processing single bets with single model...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,MODEL PROB,EDGE,EV%,KELLY_FRACTION,KELLY_DOLLARS,CONFIDENCE INTERVAL,INTERVAL WIDTH,SIGMA,SIGMA FLAG,EXPECTED ROI,SIMULATION_METHOD
0,Keldon Johnson,Bovada,player_points,13.5,180,Over,14.12,0,0.545,0.455,0.357,0.545,0.188,5.25,0.292,4.50,"(2.4, 25.9)",23.51,6.00,Med,0.53,Monte Carlo
1,Goga Bitadze,BetMGM,player_points,5.5,100,Over,8.51,1,0.737,0.263,0.500,0.737,0.237,4.73,0.473,2.50,"(0.0, 19.5)",19.48,5.60,Med,0.47,Monte Carlo
2,Keldon Johnson,Bovada,player_points,12.5,135,Over,14.12,0,0.621,0.379,0.426,0.621,0.196,4.60,0.341,3.38,"(2.4, 25.9)",23.51,6.00,Med,0.46,Monte Carlo
3,Keldon Johnson,Bovada,player_points,11.5,105,Over,14.12,1,0.696,0.304,0.488,0.696,0.208,4.26,0.406,2.62,"(2.4, 25.9)",23.51,6.00,Med,0.43,Monte Carlo
4,Taurean Prince,BetMGM,player_points,4.5,-140,Over,9.02,1,0.831,0.169,0.583,0.831,0.248,4.25,0.595,1.79,"(0.0, 19.8)",19.77,5.49,Med,0.42,Monte Carlo


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
underdogPairs = underdogPairs[((underdogPairs['sigma_flag1'] == 'Med') & (underdogPairs['sigma_flag2'] == 'Med')) | ((underdogPairs['sigma_flag1'] == 'Low') & (underdogPairs['sigma_flag2'] == 'Low')) | ((underdogPairs['sigma_flag1'] == 'Med') & (underdogPairs['sigma_flag2'] == 'Low')) | ((underdogPairs['sigma_flag1'] == 'Low') & (underdogPairs['sigma_flag2'] == 'Med'))].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs_{today}.csv', index=False)
underdogPairs.head()

Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single posi

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Aaron Gordon,Chris Paul,15.5,4.5,24.31,9.01,over,over,0.946,0.836,0.7118,0.368,0.258,0.313,1.14,0.568,0,"(12.9, 35.7)","(0.0, 19.6)",22.74,19.57,5.80,5.39,Med,Med,Monte Carlo
1,Sion James,Aaron Gordon,5.5,15.5,10.28,24.31,over,over,0.836,0.946,0.7120,0.258,0.368,0.313,1.14,0.568,0,"(0.0, 21.5)","(12.9, 35.7)",21.46,22.74,5.70,5.80,Med,Med,Monte Carlo
2,Jaylin Williams,Aaron Gordon,5.5,15.5,9.99,24.31,over,over,0.830,0.946,0.7066,0.252,0.368,0.310,1.12,0.560,0,"(0.0, 20.7)","(12.9, 35.7)",20.72,22.74,5.48,5.80,Med,Med,Monte Carlo
3,Corey Kispert,Aaron Gordon,6.5,15.5,10.50,24.31,over,over,0.784,0.946,0.6677,0.206,0.368,0.287,1.00,0.502,0,"(0.0, 22.2)","(12.9, 35.7)",22.22,22.74,5.98,5.80,Med,Med,Monte Carlo
4,Goga Bitadze,Aaron Gordon,5.5,15.5,8.51,24.31,over,over,0.757,0.946,0.6445,0.179,0.368,0.273,0.93,0.467,0,"(0.0, 18.5)","(12.9, 35.7)",18.48,22.74,5.09,5.80,Med,Med,Monte Carlo


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
pairsPrizepicks = pairsPrizepicks[((pairsPrizepicks['sigma_flag1'] == 'Med') & (pairsPrizepicks['sigma_flag2'] == 'Med')) | ((pairsPrizepicks['sigma_flag1'] == 'Low') & (pairsPrizepicks['sigma_flag2'] == 'Low')) | ((pairsPrizepicks['sigma_flag1'] == 'Med') & (pairsPrizepicks['sigma_flag2'] == 'Low')) | ((pairsPrizepicks['sigma_flag1'] == 'Low') & (pairsPrizepicks['sigma_flag2'] == 'Med'))].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs_{today}.csv', index=False)
pairsPrizepicks.head()

Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single posi

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Aaron Gordon,Chris Paul,15.5,4.5,24.31,9.01,over,over,0.946,0.836,0.7118,0.368,0.258,0.313,1.14,0.568,0,"(12.9, 35.7)","(0.0, 19.6)",22.74,19.57,5.80,5.39,Med,Med,Monte Carlo
1,Sion James,Aaron Gordon,5.5,15.5,10.28,24.31,over,over,0.836,0.946,0.7120,0.258,0.368,0.313,1.14,0.568,0,"(0.0, 21.5)","(12.9, 35.7)",21.46,22.74,5.70,5.80,Med,Med,Monte Carlo
2,Jaylin Williams,Aaron Gordon,5.5,15.5,9.99,24.31,over,over,0.830,0.946,0.7066,0.252,0.368,0.310,1.12,0.560,0,"(0.0, 20.7)","(12.9, 35.7)",20.72,22.74,5.48,5.80,Med,Med,Monte Carlo
3,Corey Kispert,Aaron Gordon,6.5,15.5,10.50,24.31,over,over,0.784,0.946,0.6677,0.206,0.368,0.287,1.00,0.502,0,"(0.0, 22.2)","(12.9, 35.7)",22.22,22.74,5.98,5.80,Med,Med,Monte Carlo
4,Sion James,Chris Paul,5.5,4.5,10.28,9.01,over,over,0.836,0.836,0.6289,0.258,0.258,0.258,0.89,0.443,0,"(0.0, 21.5)","(0.0, 19.6)",21.46,19.57,5.70,5.39,Med,Med,Monte Carlo


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)


underdogTrios = threeLeg[((threeLeg['sigma_flag1'] == 'Med') & (threeLeg['sigma_flag2'] == 'Med') & (threeLeg['sigma_flag3'] == 'Med')) | ((threeLeg['sigma_flag1'] == 'Low') & (threeLeg['sigma_flag2'] == 'Low') & (threeLeg['sigma_flag3'] == 'Low')) | ((threeLeg['sigma_flag1'] == 'Med') & (threeLeg['sigma_flag2'] == 'Low') & (threeLeg['sigma_flag3'] == 'Low')) | ((threeLeg['sigma_flag1'] == 'Low') & (threeLeg['sigma_flag2'] == 'Med') & (threeLeg['sigma_flag3'] == 'Low')) | ((threeLeg['sigma_flag1'] == 'Low') & (threeLeg['sigma_flag2'] == 'Low') & (threeLeg['sigma_flag3'] == 'Med'))].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios_{today}.csv', index=False)
underdogTrios.head()

Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single posi

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method
0,Aaron Gordon,Bruce Brown,Chris Paul,15.5,4.5,4.5,24.31,9.24,9.01,over,over,over,0.946,0.850,0.836,0.5446,0.368,0.272,0.258,0.299,2.27,0.454,0,"(12.9, 35.7)","(0.0, 19.6)","(0.0, 19.6)",22.74,19.62,19.57,5.80,5.29,5.39,Med,Med,Med,Monte Carlo
1,Sion James,Aaron Gordon,Bruce Brown,5.5,15.5,4.5,10.28,24.31,9.24,over,over,over,0.836,0.946,0.850,0.5447,0.258,0.368,0.272,0.299,2.27,0.454,0,"(0.0, 21.5)","(12.9, 35.7)","(0.0, 19.6)",21.46,22.74,19.62,5.70,5.80,5.29,Med,Med,Med,Monte Carlo
2,Jaylin Williams,Aaron Gordon,Bruce Brown,5.5,15.5,4.5,9.99,24.31,9.24,over,over,over,0.830,0.946,0.850,0.5406,0.252,0.368,0.272,0.297,2.24,0.449,0,"(0.0, 20.7)","(12.9, 35.7)","(0.0, 19.6)",20.72,22.74,19.62,5.48,5.80,5.29,Med,Med,Med,Monte Carlo
3,Sion James,Aaron Gordon,Chris Paul,5.5,15.5,4.5,10.28,24.31,9.01,over,over,over,0.836,0.946,0.836,0.5356,0.258,0.368,0.258,0.295,2.21,0.443,0,"(0.0, 21.5)","(12.9, 35.7)","(0.0, 19.6)",21.46,22.74,19.57,5.70,5.80,5.39,Med,Med,Med,Monte Carlo
4,Sion James,Jaylin Williams,Aaron Gordon,5.5,5.5,15.5,10.28,9.99,24.31,over,over,over,0.836,0.830,0.946,0.5317,0.258,0.252,0.368,0.293,2.19,0.438,0,"(0.0, 21.5)","(0.0, 20.7)","(12.9, 35.7)",21.46,20.72,22.74,5.70,5.48,5.80,Med,Med,Med,Monte Carlo


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg[((threeLeg['sigma_flag1'] == 'Med') & (threeLeg['sigma_flag2'] == 'Med') & (threeLeg['sigma_flag3'] == 'Med')) | ((threeLeg['sigma_flag1'] == 'Low') & (threeLeg['sigma_flag2'] == 'Low') & (threeLeg['sigma_flag3'] == 'Low')) | ((threeLeg['sigma_flag1'] == 'Med') & (threeLeg['sigma_flag2'] == 'Low') & (threeLeg['sigma_flag3'] == 'Low')) | ((threeLeg['sigma_flag1'] == 'Low') & (threeLeg['sigma_flag2'] == 'Med') & (threeLeg['sigma_flag3'] == 'Low')) | ((threeLeg['sigma_flag1'] == 'Low') & (threeLeg['sigma_flag2'] == 'Low') & (threeLeg['sigma_flag3'] == 'Med'))].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios_{today}.csv', index=False)
triosPrizepicks.head()

Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single positional indexer is out-of-bounds
Error getting prediction for Jaxson Hayes: single posi

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method
0,Sion James,Jaylin Williams,Aaron Gordon,5.5,5.5,15.5,10.28,9.99,24.31,over,over,over,0.798,0.800,0.937,0.4852,0.220,0.222,0.359,0.267,1.91,0.382,0,"(0.0, 21.5)","(0.0, 20.7)","(12.9, 35.7)",21.46,20.72,22.74,5.70,5.48,5.80,Med,Med,Med,Monte Carlo
1,Sion James,Aaron Gordon,Chris Paul,5.5,15.5,4.5,10.28,24.31,9.01,over,over,over,0.798,0.940,0.791,0.4807,0.220,0.362,0.213,0.265,1.88,0.377,0,"(0.0, 21.5)","(12.9, 35.7)","(0.0, 19.6)",21.46,22.74,19.57,5.70,5.80,5.39,Med,Med,Med,Monte Carlo
2,Jaylin Williams,Aaron Gordon,Chris Paul,5.5,15.5,4.5,9.99,24.31,9.01,over,over,over,0.792,0.940,0.791,0.4771,0.214,0.362,0.213,0.263,1.86,0.372,0,"(0.0, 20.7)","(12.9, 35.7)","(0.0, 19.6)",20.72,22.74,19.57,5.48,5.80,5.39,Med,Med,Med,Monte Carlo
3,Sion James,Corey Kispert,Aaron Gordon,5.5,6.5,15.5,10.28,10.50,24.31,over,over,over,0.798,0.753,0.937,0.4567,0.220,0.175,0.359,0.252,1.74,0.348,0,"(0.0, 21.5)","(0.0, 22.2)","(12.9, 35.7)",21.46,22.22,22.74,5.70,5.98,5.80,Med,Med,Med,Monte Carlo
4,Corey Kispert,Jaylin Williams,Aaron Gordon,6.5,5.5,15.5,10.50,9.99,24.31,over,over,over,0.749,0.800,0.937,0.4553,0.171,0.222,0.359,0.251,1.73,0.346,0,"(0.0, 22.2)","(0.0, 20.7)","(12.9, 35.7)",22.22,20.72,22.74,5.98,5.48,5.80,Med,Med,Med,Monte Carlo


In [10]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'LAST {window}'] = hits
        results[f'HIT RATE % LAST {window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Franz Wagner,Over,21.5,-137,2025-10-30,2025-10-30T21:22:09Z
2,PrizePicks,player_points,Paolo Banchero,Over,27.0,-137,2025-10-30,2025-10-30T21:22:09Z
4,PrizePicks,player_points,LaMelo Ball,Over,26.0,-137,2025-10-30,2025-10-30T21:22:09Z
6,PrizePicks,player_points,Miles Bridges,Over,20.0,-137,2025-10-30,2025-10-30T21:22:09Z
8,PrizePicks,player_points,Desmond Bane,Over,19.5,-137,2025-10-30,2025-10-30T21:22:09Z
...,...,...,...,...,...,...,...,...
2413,PrizePicks,player_blocks_steals,Christian Braun,Over,1.5,-137,2025-11-01,2025-10-30T21:26:18Z
2415,PrizePicks,player_blocks_steals,Jerami Grant,Over,1.5,-137,2025-11-01,2025-10-30T21:26:18Z
2417,PrizePicks,player_blocks_steals,Brook Lopez,Over,1.5,-137,2025-11-01,2025-10-30T21:26:29Z
2419,PrizePicks,player_blocks_steals,Yves Missi,Over,1.5,-137,2025-11-01,2025-10-30T21:26:29Z


In [11]:



line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 84 records for player_points to player_points.csv
Saved 42 records for player_rebounds to player_rebounds.csv
Saved 27 records for player_assists to player_assists.csv
Saved 18 records for player_threes to player_threes.csv
Saved 6 records for player_blocks to player_blocks.csv
Saved 13 records for player_steals to player_steals.csv
Saved 13 records for player_field_goals to player_field_goals.csv
Saved 20 records for player_frees_made to player_frees_made.csv
Saved 23 records for player_frees_attempts to player_frees_attempts.csv
Saved 81 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 79 records for player_points_rebounds to player_points_rebounds.csv
Saved 78 records for player_points_assists to player_points_assists.csv
Saved 50 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 13 records for player_turnovers to player_turnovers.csv
Saved 20 records for player_blocks_steals to player_blocks_steals.csv

All category